# 01 — Data Preparation
### Chicago Housing Market | Listing Risk Prediction

This is the **first step** of every data science project. Before we can explore, model, or predict anything, we need clean, well-structured data.

**What this notebook does:**
1. Load the raw Redfin metro market dataset
2. Filter it down to Chicago
3. Select and rename the columns we need
4. Fix data types
5. Handle any missing values
6. Save the clean data to a SQLite database table for use in all other notebooks

**Output:** `chicago_housing` table in `sql/housing.db`

### What are we doing in this cell?

We import the libraries we need. For data preparation we only need:
- **`pandas`** — to load and manipulate the data table
- **`numpy`** — for numeric operations
- **`os`** — to check that file paths exist before saving

In [2]:
import pandas as pd
import numpy as np
import os

print('Libraries loaded.')


Libraries loaded.


---
## 1. Load Raw Data

### What are we doing in this cell?

The raw data comes from **Redfin's Metro Market Tracker** — a publicly available dataset that tracks housing market conditions (prices, inventory, days on market, etc.) for hundreds of US metro areas every month.

The file is a **TSV (Tab-Separated Values)** file — just like a CSV but columns are separated by a tab character instead of a comma. We tell pandas this with `sep='\t'`.

We set `low_memory=False` because the file is large (~560,000 rows) and has mixed data types — this prevents pandas from guessing incorrectly.

In [3]:
RAW_PATH = 'redfin_metro_market_tracker.tsv'

df_raw = pd.read_csv(RAW_PATH, sep='\t', low_memory=False)

print(f'Raw dataset shape: {df_raw.shape}')
print(f'Columns ({len(df_raw.columns)}): {list(df_raw.columns)}')
df_raw.head(3)

Raw dataset shape: (561266, 58)
Columns (58): ['PERIOD_BEGIN', 'PERIOD_END', 'PERIOD_DURATION', 'REGION_TYPE', 'REGION_TYPE_ID', 'TABLE_ID', 'IS_SEASONALLY_ADJUSTED', 'REGION', 'CITY', 'STATE', 'STATE_CODE', 'PROPERTY_TYPE', 'PROPERTY_TYPE_ID', 'MEDIAN_SALE_PRICE', 'MEDIAN_SALE_PRICE_MOM', 'MEDIAN_SALE_PRICE_YOY', 'MEDIAN_LIST_PRICE', 'MEDIAN_LIST_PRICE_MOM', 'MEDIAN_LIST_PRICE_YOY', 'MEDIAN_PPSF', 'MEDIAN_PPSF_MOM', 'MEDIAN_PPSF_YOY', 'MEDIAN_LIST_PPSF', 'MEDIAN_LIST_PPSF_MOM', 'MEDIAN_LIST_PPSF_YOY', 'HOMES_SOLD', 'HOMES_SOLD_MOM', 'HOMES_SOLD_YOY', 'PENDING_SALES', 'PENDING_SALES_MOM', 'PENDING_SALES_YOY', 'NEW_LISTINGS', 'NEW_LISTINGS_MOM', 'NEW_LISTINGS_YOY', 'INVENTORY', 'INVENTORY_MOM', 'INVENTORY_YOY', 'MONTHS_OF_SUPPLY', 'MONTHS_OF_SUPPLY_MOM', 'MONTHS_OF_SUPPLY_YOY', 'MEDIAN_DOM', 'MEDIAN_DOM_MOM', 'MEDIAN_DOM_YOY', 'AVG_SALE_TO_LIST', 'AVG_SALE_TO_LIST_MOM', 'AVG_SALE_TO_LIST_YOY', 'SOLD_ABOVE_LIST', 'SOLD_ABOVE_LIST_MOM', 'SOLD_ABOVE_LIST_YOY', 'PRICE_DROPS', 'PRICE_DROPS_M

,PERIOD_BEGIN,PERIOD_END,PERIOD_DURATION,REGION_TYPE,REGION_TYPE_ID,TABLE_ID,IS_SEASONALLY_ADJUSTED,REGION,CITY,STATE,...,SOLD_ABOVE_LIST_YOY,PRICE_DROPS,PRICE_DROPS_MOM,PRICE_DROPS_YOY,OFF_MARKET_IN_TWO_WEEKS,OFF_MARKET_IN_TWO_WEEKS_MOM,OFF_MARKET_IN_TWO_WEEKS_YOY,PARENT_METRO_REGION,PARENT_METRO_REGION_METRO_CODE,LAST_UPDATED
0,2019-08-01,2019-08-31,30,metro,-2,35820,False,"North Platte, NE metro area",NaN,NaN,...,-0.096599,NaN,NaN,NaN,0.044444,0.022222,0.044444,"North Platte, NE",35820,2026-01-12 14:43:38.223 Z
1,2020-09-01,2020-09-30,30,metro,-2,24100,False,"Gloversville, NY metro area",NaN,NaN,...,0.077051,0.266667,0.014286,0.015819,0.156250,-0.157003,-0.063750,"Gloversville, NY",24100,2026-01-12 14:43:38.223 Z
2,2017-04-01,2017-04-30,30,metro,-2,41780,False,"Sandusky, OH metro area",NaN,NaN,...,0.000000,NaN,NaN,NaN,0.000000,0.000000,0.000000,"Sandusky, OH",41780,2026-01-12 14:43:38.223 Z


In [4]:
df_raw.columns

Index(['PERIOD_BEGIN', 'PERIOD_END', 'PERIOD_DURATION', 'REGION_TYPE',
       'REGION_TYPE_ID', 'TABLE_ID', 'IS_SEASONALLY_ADJUSTED', 'REGION',
       'CITY', 'STATE', 'STATE_CODE', 'PROPERTY_TYPE', 'PROPERTY_TYPE_ID',
       'MEDIAN_SALE_PRICE', 'MEDIAN_SALE_PRICE_MOM', 'MEDIAN_SALE_PRICE_YOY',
       'MEDIAN_LIST_PRICE', 'MEDIAN_LIST_PRICE_MOM', 'MEDIAN_LIST_PRICE_YOY',
       'MEDIAN_PPSF', 'MEDIAN_PPSF_MOM', 'MEDIAN_PPSF_YOY', 'MEDIAN_LIST_PPSF',
       'MEDIAN_LIST_PPSF_MOM', 'MEDIAN_LIST_PPSF_YOY', 'HOMES_SOLD',
       'HOMES_SOLD_MOM', 'HOMES_SOLD_YOY', 'PENDING_SALES',
       'PENDING_SALES_MOM', 'PENDING_SALES_YOY', 'NEW_LISTINGS',
       'NEW_LISTINGS_MOM', 'NEW_LISTINGS_YOY', 'INVENTORY', 'INVENTORY_MOM',
       'INVENTORY_YOY', 'MONTHS_OF_SUPPLY', 'MONTHS_OF_SUPPLY_MOM',
       'MONTHS_OF_SUPPLY_YOY', 'MEDIAN_DOM', 'MEDIAN_DOM_MOM',
       'MEDIAN_DOM_YOY', 'AVG_SALE_TO_LIST', 'AVG_SALE_TO_LIST_MOM',
       'AVG_SALE_TO_LIST_YOY', 'SOLD_ABOVE_LIST', 'SOLD_ABOVE_LIST_MOM',
 

---
## 2. Filter to Chicago

### What are we doing in this cell?

The raw dataset covers **every metro area in the United States** — we only want one consistent Chicago monthly series.

We select the Chicago metro area's **All Residential** data that is **not seasonally adjusted**. This gives us one row per month and preserves the natural seasonal pattern for the model to learn.

In [5]:
chicago = df_raw[
    (df_raw['REGION'] == 'Chicago, IL metro area') &
    (df_raw['PROPERTY_TYPE'] == 'All Residential') &
    (df_raw['IS_SEASONALLY_ADJUSTED'] == False)
].copy()

print(f'Chicago rows: {len(chicago)}')
print(f'\nUnique regions:          {chicago["REGION"].unique()}')
print(f'Unique property types:   {chicago["PROPERTY_TYPE"].unique()}')
print(f'Seasonally adjusted?     {chicago["IS_SEASONALLY_ADJUSTED"].unique()}')
print(f'Date range: {chicago["PERIOD_BEGIN"].min()}  →  {chicago["PERIOD_BEGIN"].max()}')
chicago.head()

Chicago rows: 168

Unique regions:          ['Chicago, IL metro area']
Unique property types:   ['All Residential']
Seasonally adjusted?     [False]
Date range: 2012-01-01  →  2025-12-01


,PERIOD_BEGIN,PERIOD_END,PERIOD_DURATION,REGION_TYPE,REGION_TYPE_ID,TABLE_ID,IS_SEASONALLY_ADJUSTED,REGION,CITY,STATE,...,SOLD_ABOVE_LIST_YOY,PRICE_DROPS,PRICE_DROPS_MOM,PRICE_DROPS_YOY,OFF_MARKET_IN_TWO_WEEKS,OFF_MARKET_IN_TWO_WEEKS_MOM,OFF_MARKET_IN_TWO_WEEKS_YOY,PARENT_METRO_REGION,PARENT_METRO_REGION_METRO_CODE,LAST_UPDATED
156,2019-10-01,2019-10-31,30,metro,-2,16984,False,"Chicago, IL metro area",NaN,NaN,...,-0.000979,0.261716,0.002749,-0.016495,0.249619,-0.026897,-0.011645,"Chicago, IL",16984,2026-01-12 14:43:38.223 Z
2365,2019-09-01,2019-09-30,30,metro,-2,16984,False,"Chicago, IL metro area",NaN,NaN,...,-0.006672,0.258966,0.000016,-0.004535,0.276516,-0.021471,-0.018306,"Chicago, IL",16984,2026-01-12 14:43:38.223 Z
2733,2024-08-01,2024-08-31,30,metro,-2,16984,False,"Chicago, IL metro area",NaN,NaN,...,-0.033532,0.178604,0.007639,0.003364,0.455224,-0.032560,-0.030018,"Chicago, IL",16984,2026-01-12 14:43:38.223 Z
4049,2015-06-01,2015-06-30,30,metro,-2,16984,False,"Chicago, IL metro area",NaN,NaN,...,-0.019088,0.291556,0.020536,0.001802,0.293726,-0.022835,-0.064565,"Chicago, IL",16984,2026-01-12 14:43:38.223 Z
6951,2023-02-01,2023-02-28,30,metro,-2,16984,False,"Chicago, IL metro area",NaN,NaN,...,-0.074056,0.117347,-0.021837,0.037266,0.430550,0.105420,-0.080884,"Chicago, IL",16984,2026-01-12 14:43:38.223 Z


---
## 3. Select & Rename Columns

### What are we doing in this cell?

The raw dataset has **57 columns** — month-over-month changes, year-over-year changes, list prices, pending sales, and more. For this project we only need 8 core market indicators that will be our features and targets.

| Column kept | What it measures |
|------------|------------------|
| `PERIOD_BEGIN` | The start date of the monthly period |
| `MEDIAN_SALE_PRICE` | Median price of homes that sold |
| `HOMES_SOLD` | Number of homes sold |
| `NEW_LISTINGS` | New homes listed for sale |
| `INVENTORY` | Total active listings |
| `MONTHS_OF_SUPPLY` | How many months to sell all inventory at current pace |
| `MEDIAN_DOM` | Median days a home sat on the market before selling |
| `PRICE_DROPS` | Fraction of listings that cut their asking price |

We also drop any rows where the non-date columns are all missing (sometimes Redfin has placeholder rows).

In [6]:
KEEP_COLS = [
    'PERIOD_BEGIN', 'MEDIAN_SALE_PRICE', 'HOMES_SOLD', 'NEW_LISTINGS',
    'INVENTORY', 'MONTHS_OF_SUPPLY', 'MEDIAN_DOM', 'PRICE_DROPS'
]

df = chicago[KEEP_COLS].copy()

print(f'Shape after column selection: {df.shape}')
df.head()

Shape after column selection: (168, 8)


,PERIOD_BEGIN,MEDIAN_SALE_PRICE,HOMES_SOLD,NEW_LISTINGS,INVENTORY,MONTHS_OF_SUPPLY,MEDIAN_DOM,PRICE_DROPS
156,2019-10-01,240000.0,7747.0,9064.0,35508.0,4.6,76.0,0.261716
2365,2019-09-01,242500.0,7589.0,10058.0,37557.0,4.9,71.0,0.258966
2733,2024-08-01,351500.0,7383.0,8290.0,19798.0,2.7,50.0,0.178604
4049,2015-06-01,234000.0,11272.0,13507.0,37293.0,3.3,39.0,0.291556
6951,2023-02-01,290000.0,4301.0,6548.0,18799.0,4.4,76.0,0.117347


---
## 4. Fix Data Types

### What are we doing in this cell?

When pandas loads a CSV/TSV, it tries to guess the data type of each column. Sometimes it gets it wrong — for example, a number column might be read as text (`object`) if there are any non-numeric characters like `NA` or `--` mixed in.

We fix this by:
1. **Parsing `PERIOD_BEGIN` as a date** — so Python understands it as a timeline, not just text
2. **Casting all numeric columns to `float64`** using `pd.to_numeric(..., errors='coerce')` — the `errors='coerce'` part means "if a value can't be converted, replace it with NaN instead of crashing"

In [7]:
df['PERIOD_BEGIN'] = pd.to_datetime(df['PERIOD_BEGIN'])

NUMERIC_COLS = ['MEDIAN_SALE_PRICE', 'HOMES_SOLD', 'NEW_LISTINGS',
                'INVENTORY', 'MONTHS_OF_SUPPLY', 'MEDIAN_DOM', 'PRICE_DROPS']
for col in NUMERIC_COLS:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print('Data types after fixing:')
print(df.dtypes)
print(f'\nDate range: {df["PERIOD_BEGIN"].min().date()}  →  {df["PERIOD_BEGIN"].max().date()}')

Data types after fixing:
PERIOD_BEGIN         datetime64[ns]
MEDIAN_SALE_PRICE           float64
HOMES_SOLD                  float64
NEW_LISTINGS                float64
INVENTORY                   float64
MONTHS_OF_SUPPLY            float64
MEDIAN_DOM                  float64
PRICE_DROPS                 float64
dtype: object

Date range: 2012-01-01  →  2025-12-01


---
## 5. Handle Missing Values

### What are we doing in this cell?

Now that we've forced numeric columns to their correct type, any values that couldn't be converted became `NaN` (Not a Number — Python's way of saying "empty cell").

We check how many missing values exist per column. Then:
- If the missingness is small (< 5% of rows), we can **drop** those rows
- If a whole row is empty except for the date, we also drop it

We then print a clean summary to confirm the data is complete.

In [8]:
print('Missing values before cleaning:')
print(df.isnull().sum())

# Drop rows where all numeric columns are NaN
df.dropna(subset=NUMERIC_COLS, how='all', inplace=True)

# Drop rows where more than half the numeric columns are missing
df.dropna(thresh=len(NUMERIC_COLS) // 2 + 2, inplace=True)

print(f'\nMissing values after cleaning:')
print(df.isnull().sum())
print(f'\nRows remaining: {len(df)}')

Missing values before cleaning:
PERIOD_BEGIN         0
MEDIAN_SALE_PRICE    0
HOMES_SOLD           0
NEW_LISTINGS         0
INVENTORY            0
MONTHS_OF_SUPPLY     0
MEDIAN_DOM           0
PRICE_DROPS          0
dtype: int64

Missing values after cleaning:
PERIOD_BEGIN         0
MEDIAN_SALE_PRICE    0
HOMES_SOLD           0
NEW_LISTINGS         0
INVENTORY            0
MONTHS_OF_SUPPLY     0
MEDIAN_DOM           0
PRICE_DROPS          0
dtype: int64

Rows remaining: 168


---
## 6. Sort & Save to Database

### What are we doing in this cell?

We sort the data by date (oldest first) and reset the row index. Sorting is important for time-series analysis — later notebooks will use the row order to compute lag features and rolling averages.

Then we save the cleaned DataFrame to a SQLite database table called `chicago_housing` in `sql/housing.db`. This database is where all subsequent notebooks will load their data from — they don't need to touch the raw TSV or CSV files again.

In [9]:
df = df.sort_values('PERIOD_BEGIN').reset_index(drop=True)

import sqlite3

# Save to database instead of CSV
conn = sqlite3.connect('../sql/housing.db')
df.to_sql('chicago_housing', conn, if_exists='replace', index=False)
conn.close()

print(f'Saved to: chicago_housing table in housing.db')
print(f'Final shape: {df.shape}')
print(f'Date range: {df["PERIOD_BEGIN"].min().date()}  →  {df["PERIOD_BEGIN"].max().date()}')
df.head()

Saved to: chicago_housing table in housing.db
Final shape: (168, 8)
Date range: 2012-01-01  →  2025-12-01


,PERIOD_BEGIN,MEDIAN_SALE_PRICE,HOMES_SOLD,NEW_LISTINGS,INVENTORY,MONTHS_OF_SUPPLY,MEDIAN_DOM,PRICE_DROPS
0,2012-01-01,140000.0,3983.0,8123.0,39496.0,9.9,97.0,0.260128
1,2012-02-01,135000.0,4314.0,9637.0,39976.0,9.3,100.0,0.253127
2,2012-03-01,150000.0,5730.0,11617.0,40661.0,7.1,102.0,0.269005
3,2012-04-01,156500.0,6041.0,11111.0,40751.0,6.7,87.0,0.265834
4,2012-05-01,170000.0,7273.0,11190.0,40555.0,5.6,70.0,0.272765


---
## Summary

| Step | What we did | Result |
|------|------------|--------|
| Load | Read the Redfin TSV (560k+ rows, 57 columns) | Raw DataFrame |
| Filter | Kept only Chicago metro rows | Chicago subset |
| Select | Kept 8 relevant columns | Focused table |
| Fix types | Parsed dates, cast numerics | Correct dtypes |
| Clean | Dropped empty rows | No junk rows |
| Sort & save | Ordered by date, saved to CSV | `chicago_housing_data.csv` |

**Next step →** Open `02_exploratory_analysis.ipynb` to explore and understand the data visually.